In [1]:
import pandas as pd
import numpy as np


from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset, DatasetStats
from evidently import DataDefinition, Dataset
from evidently import BinaryClassification
from evidently.tests import *
from evidently.metrics import *



# Problem Statement

We trained our model with the red wines dataset!

Let's assume that our customers want to classify both red and white wines with our model.

Let's generate an EvidentlyAI report to see the change.

In [2]:
df_red = pd.read_csv("../data/winequality-red.csv", sep=";")
df_white = pd.read_csv("../data/winequality-white.csv", sep=";")

df_red["target"] = np.where(df_red["quality"] < 7, 0, 1)
df_white["target"] = np.where(df_white["quality"] < 7, 0, 1)

df_red.drop("quality", axis=1)
df_white.drop("quality", axis=1)

reference = df_red.drop_duplicates()
current = pd.concat([df_red, df_white])

In [3]:
# This needs the API running

# Let's get the pediction for all datapoints!
import requests

# Assuming `current` is your DataFrame
url = "http://localhost:8000/model/predict"
headers = {"accept": "application/json", "Content-Type": "application/json"}

for df in [reference, current]:
    predictions = []
    for idx, row in df.iterrows():
        # Convert row to dict, only include feature columns
        payload = row.to_dict()
        
        # Send POST request
        response = requests.post(url, headers=headers, json=payload)
        
        if response.status_code == 200:
            predictions.append(response.json()["predicted_quality"])  # store API output
        else:
            print(f"Failed for row {idx}: {response.status_code}, {response.text}")
            predictions.append(None)

    # Optionally, add predictions to your DataFrame
    df['prediction'] = predictions


In [7]:
data_def = DataDefinition(
    classification=[BinaryClassification(target="target", prediction_labels="prediction")]
)

ref_ds = Dataset.from_pandas(reference, data_definition=data_def)
cur_ds = Dataset.from_pandas(current,   data_definition=data_def)


report = Report(metrics=[
    DatasetStats(),
    DuplicatedRowCount(),
    AlmostDuplicatedColumnsCount(),
    AlmostConstantColumnsCount(),
])

dataset_report_snapshot = report.run(reference_data=reference, current_data=current)

In [10]:
with open("dataset_report.html", "w", encoding="utf-8") as f:
    f.write(dataset_report_snapshot.get_html_str(as_iframe=False))

In [11]:
report = Report(metrics=[
    DataDriftPreset(),
])

drift_report_snapshot = report.run(reference_data=reference, current_data=current)

with open("data_drift_report.html", "w", encoding="utf-8") as f:
    f.write(drift_report_snapshot.get_html_str(as_iframe=False))